In [2]:
!export LANGCHAIN_TRACING_V2="true"
!export LANGCHAIN_API_KEY="..."

# Starting the llama2 server and simple chain

In [3]:
from langchain_community.llms import Ollama
llm = Ollama(model="llama2")
llm.invoke("how can langsmith help with testing?")

"\nLangSmith is an AI-powered language model that can assist with testing in various ways:\n\n1. Automated Testing: LangSmith can be used to generate test cases based on the API documentation of a system, allowing for automated testing of the system's functionality. This can save time and resources compared to manual testing methods.\n2. Code Review: LangSmith can analyze code to identify potential issues or bugs, providing feedback on areas that need improvement. This can help developers identify and fix problems earlier in the development process.\n3. Test Data Generation: LangSmith can generate test data based on a system's requirements, allowing for more comprehensive testing of the system's functionality. This can be particularly useful when testing complex systems with many variables.\n4. Regression Testing: LangSmith can assist with regression testing by generating test cases that cover the changes made to a system since the last release. This ensures that the new features or ch

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are world class technical documentation writer."),
    ("user", "{input}")
])
output_parser = StrOutputParser()
chain = prompt | llm | output_parser

chain.invoke({"input": "how can langsmith help with testing?"})




"\nAs a world-class technical documentation writer, I must say that LingSMITH is an incredible tool for testing and documenting various aspects of software systems. Here are some ways in which LingSMITH can help with testing:\n\n1. Automated Testing: LingSMITH can be used to automate the testing process by generating test cases based on the documentation. This helps to ensure that all possible scenarios are covered, reducing the likelihood of errors and bugs.\n2. Code Generation: LingSMITH can generate code snippets based on the documentation, which can then be used to test various aspects of the software system. This saves time and effort compared to manually writing test cases.\n3. Test Data Generation: LingSMITH can also generate test data based on the documentation, which can be used to test the software system's behavior under different conditions. This helps to ensure that the software is robust and can handle a wide range of inputs.\n4. Documentation Validation: LingSMITH can va

# Basic Retrieval

Load the data from the web

In [5]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.smith.langchain.com/user_guide")
docs = loader.load()

Setup a vector database (FAISS) and embedd using ollama

In [7]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

embeddings = OllamaEmbeddings()
text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(docs)
vector = FAISS.from_documents(documents, embeddings)


Define the context retrieval template

In [8]:
from langchain.chains.combine_documents import create_stuff_documents_chain

prompt = ChatPromptTemplate.from_template("""Answer the following question based only on the provided context:

<context>
{context}
</context>

Question: {input}""")

document_chain = create_stuff_documents_chain(llm, prompt)

In [10]:
from langchain_core.documents import Document
from langchain.chains import create_retrieval_chain

retriever = vector.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)
response = retrieval_chain.invoke({"input": "how can langsmith help with testing?"})
print(response["answer"])


Based on the provided context, LangSmith can help with testing in several ways:

1. Testing LLM applications: LangSmith allows developers to create test cases and run them on their LLM applications. This helps identify any issues or regressions in the application's performance.
2. Annotating traces: LangSmith supports annotating traces, which allows users to closely inspect interesting traces and annotate them with respect to different criteria. This helps identify areas where the application is breaking down or producing unexpected results.
3. Monitoring key data points: LangSmith provides monitoring charts that allow users to track key metrics over time. This helps identify any issues or regressions in the application's performance at scale.
4. Debugging: When things go wrong (e.g., unexpected end result, infinite agent loop, slower than expected execution), LangSmith gives clear visibility and debugging information at each step of an LLM sequence, making it easier to identify and r

# Conversational Agent

Lets use the conversation as context

In [13]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage


prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    ("user", "Given the above conversation, generate a search query to look up in order to get information relevant to the conversation")
])
retriever_chain = create_history_aware_retriever(llm, retriever, prompt)

chat_history = [HumanMessage(content="Can LangSmith help test my LLM applications?"), AIMessage(content="Yes!")]
retriever_chain.invoke({
    "chat_history": chat_history,
    "input": "Tell me how"
})

[Document(page_content='Skip to main content\uf8ffü¶úÔ∏è\uf8ffüõ†Ô∏è LangSmith DocsLangChain Python DocsLangChain JS/TS DocsLangSmith API DocsSearchGo to AppLangSmithUser GuideSetupPricing (Coming Soon)Self-HostingTracingEvaluationMonitoringPrompt HubUser GuideOn this pageLangSmith User GuideLangSmith is a platform for LLM application development, monitoring, and testing. In this guide, we‚Äôll highlight the breadth of workflows LangSmith supports and how they fit into each stage of the application development lifecycle. We hope this will inform users how to best utilize this powerful platform or give them something to consider if they‚Äôre just starting their journey.Prototyping‚ÄãPrototyping LLM applications often involves quick experimentation between prompts, model types, retrieval strategy and other parameters.\nThe ability to rapidly understand how the model is performing ‚Äî and debug where it is failing ‚Äî is incredibly important for this phase.Debugging‚ÄãWhen developing new 

Now that we have this new retriever, we can create a new chain to continue the conversation with these retrieved documents in mind.

In [14]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's questions based on the below context:\n\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
])
document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriever_chain, document_chain)

chat_history = [HumanMessage(content="Can LangSmith help test my LLM applications?"), AIMessage(content="Yes!")]
retrieval_chain.invoke({
    "chat_history": chat_history,
    "input": "Tell me how"
})

{'chat_history': [HumanMessage(content='Can LangSmith help test my LLM applications?'),
  AIMessage(content='Yes!')],
 'input': 'Tell me how',
 'context': [Document(page_content='Skip to main content\uf8ffü¶úÔ∏è\uf8ffüõ†Ô∏è LangSmith DocsLangChain Python DocsLangChain JS/TS DocsLangSmith API DocsSearchGo to AppLangSmithUser GuideSetupPricing (Coming Soon)Self-HostingTracingEvaluationMonitoringPrompt HubUser GuideOn this pageLangSmith User GuideLangSmith is a platform for LLM application development, monitoring, and testing. In this guide, we‚Äôll highlight the breadth of workflows LangSmith supports and how they fit into each stage of the application development lifecycle. We hope this will inform users how to best utilize this powerful platform or give them something to consider if they‚Äôre just starting their journey.Prototyping‚ÄãPrototyping LLM applications often involves quick experimentation between prompts, model types, retrieval strategy and other parameters.\nThe ability to r